<a href="https://colab.research.google.com/github/fidlarsyn/grokking-Deep-Learning-Chapter-11-to-15/blob/main/14_Learning_to_Write_Like_Shakespeare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Character Language Modeling**

Menggeser tingkat pemodelan bahasa dari tingkat kata (*word-level*) yang diperkenalkan pada Bab 12 dan 13 menjadi tingkat karakter (*character-level language modeling*). Model dilatih menggunakan korpus teks karya William Shakespeare (`shakespear.txt`) untuk memprediksi karakter berikutnya berdasarkan sekuens karakter yang telah diamati sebelumnya. Kosa kata (*vocabulary*) kini tidak lagi berisi ribuan kata, melainkan hanya puluhan karakter unik (huruf, angka, spasi, dan tanda baca).

# **The Need for Truncated Backpropagation**

Pada pemodelan sekuens pendek (seperti 5 kata), sinyal eror dapat dirambatkan mundur melewati seluruh langkah. Namun, korpus Shakespeare memiliki lebih dari 100.000 karakter. Melakukan *backpropagation* penuh dari karakter ke-100.000 mundur hingga karakter pertama tidak memungkinkan secara komputasi dan memori.

# **Truncated Backpropagation**

**Truncated Backpropagation** adalah teknik standar industri di mana alur *backpropagation* dibatasi hanya berjalan mundur hingga sejauh jumlah langkah tertentu. Parameter panjang perambatan mundur ini disebut `bptt` (*backpropagation through time*), yang biasanya diatur antara 16 hingga 64 *timestep*. Pemotongan gradien ini menghemat memori, meskipun secara teoritis membatasi rentang memori gradien langsung model.

# **A Sample of the Output &amp; Temperature Sampling**

Setelah model dilatih, karakter baru digenerasi secara berurutan dengan memasukkan prediksi karakter sebelumnya sebagai masukan untuk *timestep* berikutnya. Untuk mengontrol tingkat variasi sampel teks yang dihasilkan, digunakan **temperature sampling**: mengalikan keluaran mentah sebelum *softmax* dengan nilai skala tertentu agar distribusi probabilitas menjadi lebih tegas (*greedy*) atau lebih acak/kreatif.

# **Vanishing and Exploding Gradients**

Masalah fundamental pada Vanilla RNN adalah fenomena **vanishing** dan **exploding gradients** saat melatih sekuens panjang. Karena sinyal eror harus melewati perkalian matriks bobot dan fungsi aktivasi non-linier berulang kali pada setiap *timestep*:
* **Vanishing Gradients**: Gradien mengecil secara eksponensial mendekati angka 0, menyebabkan model tidak mampu mempelajari hubungan jangka panjang.
* **Exploding Gradients**: Gradien membesar secara eksponensial hingga mengalami *overshooting* atau menghasilkan angka tak hingga (`NaN`), yang merusak stabilitas pelatihan.

# **Long Short-Term Memory (LSTM) Cells &amp; The Gated Copy Trick**

LSTM diciptakan untuk mengatasi masalah vanishing/exploding gradients melalui mekanisme gated copy trick. LSTM memisahkan status memori internal (cell state $c$) dari status tersembunyi keluaran (hidden state $h$). Status sel $c$ diperbarui secara aditif (penjumlahan/pengurangan) tanpa melalui perkalian matriks berulang atau fungsi aktivasi non-linier langsung, sehingga gradien dapat merambat lurus melewati waktu tanpa mengalami degradasi nilai.

# **Some Intuition About LSTM Gates**

Mekanisme LSTM diatur oleh tiga pintu gerbang (gates) biner bertingkat yang bernilai antara 0 dan 1 (menggunakan aktivasi Sigmoid):
* **Forget Gate ($f$)**: Mengontrol seberapa banyak informasi pada memori lama (cell state sebelumnya) yang harus dipertahankan atau dibuang.
* **Input Gate ($i$)**: Mengontrol seberapa banyak informasi baru dari masukan saat ini yang akan dimasukkan ke dalam status sel.
* **Output Gate ($o$)**: Mengontrol porsi dari status sel yang diekspos sebagai status tersembunyi keluaran (hidden state $h$).
* **Cell Update ($u$)**: Vektor kandidat informasi baru (menggunakan aktivasi Tanh) yang ditambahkan ke status sel.

# **Upgrading, Training, and Tuning the LSTM Character Language Model**

Mengganti RNNCell menjadi LSTMCell pada framework autograd. Proses pelatihan melibatkan pengelolaan dua status per timestep (status sel $c$ dan status tersembunyi $h$). Penyesuaian parameter (hyperparameter tuning) seperti penyelarasan nilai alpha (learning rate), batch_size, dan bptt sangat krusial untuk memastikan konvergensi model pada teks Shakespeare.

In [ ]:
import sys, random, math
import numpy as np

np.random.seed(0)

# 1. Membaca Korpus Shakespeare & Pemetaan Karakter ke Indeks
# (Pastikan file 'shakespear.txt' tersedia di direktori kerja)
f = open('shakespear.txt', 'r')
raw = f.read()
f.close()

vocab = list(set(raw))
word2index = {\}
for i, char in enumerate(vocab):
    word2index[char] = i

indices = np.array(list(map(lambda x: word2index[x], raw)))

# 2. Pengaturan Truncated BPTT Batching
batch_size = 32
bptt = 25
n_batches = int(indices.shape / batch_size)

trimmed_indices = indices[:n_batches * batch_size]
batched_indices = trimmed_indices.reshape(batch_size, n_batches).transpose()

input_batched_indices = batched_indices[0:-1]
target_batched_indices = batched_indices[1:]

n_bptt = int((n_batches - 1) / bptt)
input_batches = input_batched_indices[:n_bptt * bptt].reshape(n_bptt, bptt, batch_size)
target_batches = target_batched_indices[:n_bptt * bptt].reshape(n_bptt, bptt, batch_size)

# 3. Inisialisasi Model LSTM (Menggunakan Framework Autograd Bab 13)
# Note: Menggunakan abstraksi Embedding & LSTMCell dari Framework Bab 13
embed = Embedding(vocab_size=len(vocab), dim=512)
model = LSTMCell(n_inputs=512, n_hidden=512, n_output=len(vocab))
model.w_ho.weight.data *= 0  # Inisialisasi bobot output ke 0

criterion = CrossEntropyLoss()
optim = SGD(parameters=model.get_parameters() + embed.get_parameters(), alpha=0.05)

# 4. Fungsi Sampling Generasi Teks Shakespeare
def generate_sample(n=100, init_char='T'):
    s = ""
    hidden = model.init_hidden(batch_size=1)
    input_t = Tensor(np.array([word2index[init_char]]))
    for i in range(n):
        rnn_input = embed.forward(input_t)
        output, hidden = model.forward(input=rnn_input, hidden=hidden)

        # Temperature Sampling (Menajamkan Distribusi)
        output.data *= 15
        temp_dist = output.softmax()
        temp_dist /= temp_dist.sum()

        m = output.data.argmax()
        c = vocab[m]
        input_t = Tensor(np.array([m]))
        s += c
    return s

# 5. Training Loop dengan Truncated Backpropagation Through Time (BPTT)
min_loss = 1000
iterations = 10

for iter in range(iterations):
    total_loss = 0
    hidden = model.init_hidden(batch_size=batch_size)

    for batch_i in range(len(input_batches)):
        # Detach status memori agar gradien dipotong (Truncated BPTT)
        hidden = (Tensor(hidden.data, autograd=True),
                  Tensor(hidden[2].data, autograd=True))

        losses = list()
        for t in range(bptt):
            input_t = Tensor(input_batches[batch_i][t], autograd=True)
            rnn_input = embed.forward(input=input_t)
            output, hidden = model.forward(input=rnn_input, hidden=hidden)

            target_t = Tensor(target_batches[batch_i][t], autograd=True)
            batch_loss = criterion.forward(output, target_t)

            if t == 0:
                losses.append(batch_loss)
            else:
                losses.append(batch_loss + losses[-1])

        # Perambatan Mundur & Pembaruan Bobot per Blok BPTT
        loss = losses[-1]
        loss.backward()
        optim.step()

        total_loss += loss.data / bptt
        epoch_loss = np.exp(total_loss / (batch_i + 1))

        if epoch_loss < min_loss:
            min_loss = epoch_loss

        if (batch_i + 1) % 50 == 0:
            print(f"Iter: {iter\} | Batch {batch_i+1\}/{len(input_batches)\} | Min Loss: {min_loss:.3f\} | Loss: {epoch_loss:.3f\}")
            print("Sample Teks:", generate_sample(n=50, init_char='T'))

    optim.alpha *= 0.99  # Peluruhan Learning Rate